## Micrograd

In [1]:
import math

class Value:

    def __init__(self, data, _children = (), op = (), label = " "):
        self.data = data
        self.grad = 0
        self._backward = lambda: None 
        self._prev = set(_children)
        self._op = op
        self._label = label
    
    def __repr__(self):
        return f"Value(label={self._label}, data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        
        out._backward = _backward 
        return out

    def __radd__(self, other):
        return self + other

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        
        out._backward = _backward 
        return out

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * (other**-1)

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only int and float powers are supported for now"
        out = Value(self.data ** other, (self,), f"**{other}")

        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad

        out._backward = _backward
        return out

    def tanh(self):
        n = self.data
        t = (math.exp(2*n) - 1)/(math.exp(2*n) + 1)
        out = Value(t, (self, ), label="tanh")

        def _backward():
            self.grad +=  (1 - t**2) * out.grad
        
        out._backward = _backward 
        return out

    def backprop(self):
        self._backward()
        # print(f"label: {self._label}, grad: {self.grad}")
        for child in self._prev:
            child.backprop()

In [2]:
a = Value(3.0, label="a")
b = a + a
b.grad = 1
b.backprop()

In [3]:
a + 1
a * 2
2 * a
2 + a

Value(label= , data=5.0)

In [4]:
x1 = Value(2.0, label="x1")
x2 = Value(0.0, label="x2")

w1 = Value(-3.0, label="w1")
w2 = Value(1.0, label="w2")

b = Value(6.8813735870195432, label="b")

x1w1 = x1*w1
x1w1._label = "x1*w1"

x2w2 = x2*w2
x2w2._label = "x2*w2"

x1w1x2w2 = x1w1 + x2w2
x1w1x2w2._label = "w1*w1 + x2*w2"

n = x1w1x2w2 + b
n._label = "n"

o = n.tanh()
o._label = "o"

In [5]:
o.grad = 1.0

In [6]:
o.backprop()

In [7]:
import random

class Neuron:

    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for i in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        assert len(x) == len(self.w), "number of inputs is not equal to the number of weights"
        act = sum([xi * self.w[i] for i, xi in enumerate(x)]) + self.b
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]

In [8]:
x = [2.0, 3.0]
n = Neuron(len(x))
n(x)

Value(label=tanh, data=0.8247065148583019)

In [9]:
class Layer:

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        out = [neuron(x) for neuron in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

In [10]:
x = [2.0, 3.0]
la = Layer(len(x), 3)
la(x)

[Value(label=tanh, data=-0.9999510928522065),
 Value(label=tanh, data=-0.9895146743561528),
 Value(label=tanh, data=-0.5177128484482765)]

In [11]:
class MLP:

    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        out = x
        for layer in self.layers:
            out = layer(out)
        return out

    def parameters(self):
        return [p for la in self.layers for p in la.parameters()]

In [12]:
x = [2.0, 3.0, -1.0]
mlp = MLP(len(x), [4, 4, 1])
mlp(x)

Value(label=tanh, data=-0.4647398556664049)

In [13]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]

ys = [1.0, -1.0, -1.0, 1.0]

In [14]:
ypreds = [mlp(x) for x in xs]
ypreds

[Value(label=tanh, data=-0.4647398556664049),
 Value(label=tanh, data=-0.6567980347211214),
 Value(label=tanh, data=-0.8184507692561969),
 Value(label=tanh, data=-0.3881950533037279)]

In [15]:
loss = sum([(ypred - y)**2 for y, ypred in zip(ys, ypreds)])
loss._label = "loss"
loss

Value(label=loss, data=4.223296062949531)

In [16]:
loss.grad = 1.0
loss.backprop()

In [17]:
len(mlp.parameters())

41

In [18]:
STEP = 0.01

for p in mlp.parameters():
    p_old = p.data
    p.data += p.grad * (-STEP)
    print(f"parameter updated from {p_old} to {p.data}")

parameter updated from 0.36089893279116114 to -1.8397885655325783
parameter updated from -0.12112499052761438 to -2.1645274569966793
parameter updated from 0.35120505132017876 to 1.0650214382240781
parameter updated from 0.4476881265921615 to 0.21357437467369936
parameter updated from 0.8823259843802633 to 0.8745842834750379
parameter updated from -0.03686939818910262 to -0.043572995651450674
parameter updated from -0.27012432866521197 to -0.26696710634286236
parameter updated from 0.04903421655476148 to 0.047882870558015975
parameter updated from -0.7031854971937361 to -0.8611521885987585
parameter updated from 0.22849599803384413 to 0.09415603521893223
parameter updated from 0.8864987269048796 to 0.9492162543679461
parameter updated from -0.40192043626139706 to -0.41576111688296413
parameter updated from -0.4455004596888448 to -0.37649402932218945
parameter updated from -0.5718843035715606 to -0.5083171675449935
parameter updated from 0.37389506487007496 to 0.3260350997869895
paramet

In [19]:
TRAINING_ITER = 100

def train(mlp, xs, ys):
    for i in range(TRAINING_ITER):
        # forward
        ypreds = [mlp(x) for x in xs]
        loss = sum([(ypred - y)**2 for y, ypred in zip(ys, ypreds)])
        print(f"loss {loss}")
        loss._label = "loss"
        loss.grad = 1
        for p in mlp.parameters():
            p.grad = 0.0
        # backward
        loss.backprop()
        # update
        for p in mlp.parameters():
            p.data += p.grad * (-STEP)
    

In [20]:
train(mlp, xs, ys)

ypreds = [mlp(x) for x in xs]
loss = sum([(ypred - y)**2 for y, ypred in zip(ys, ypreds)])
loss

loss Value(label= , data=3.257340942564637)
loss Value(label= , data=3.019344602887912)
loss Value(label= , data=2.8775089269244605)
loss Value(label= , data=2.8397414188852856)
loss Value(label= , data=2.771757222524535)
loss Value(label= , data=2.4697594107310232)
loss Value(label= , data=1.4667680238349563)
loss Value(label= , data=1.018881362504276)
loss Value(label= , data=0.8516436829023982)
loss Value(label= , data=0.7350608278932422)
loss Value(label= , data=0.6496366869265647)
loss Value(label= , data=0.5795841512567509)
loss Value(label= , data=0.5265183013738579)
loss Value(label= , data=0.4794700943958816)
loss Value(label= , data=0.43696996202128435)
loss Value(label= , data=0.403692893657719)
loss Value(label= , data=0.37659388475099903)
loss Value(label= , data=0.3427961890840244)
loss Value(label= , data=0.31712396826542605)
loss Value(label= , data=0.3084602938951938)
loss Value(label= , data=0.2895783189384008)
loss Value(label= , data=0.2711152189492849)
loss Value(l

Value(label= , data=0.024588796914751715)

In [21]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

In [23]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]

ys = [1.0, -1.0, -1.0, 1.0]


In [25]:
from lib.mlp import MLP

mlp = MLP(len(x), [4, 4, 1])
mlp.train(xs, ys, 100, 0.01)